# Churn Prediction Model Notebook

This notebook demonstrates the end-to-end churn model workflow for the D2C churn prediction project. It covers data loading, feature preparation, leakage checks, train/validation/test split, baseline modeling, stronger modeling, evaluation, threshold selection, feature importance, and model saving.

In [49]:
import json
import os
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [50]:
ROOT_DIR = Path().resolve()
DATA_DIR = Path(os.getenv('D2C_DATA_PATH', ROOT_DIR / 'data'))
OUTPUT_DIR = ROOT_DIR
SNAPSHOT_FILE = DATA_DIR / 'rfm_modeling_snapshot.csv'
LABEL_FILE = DATA_DIR / 'churn_labels.csv'
MODEL_OUTPUT = OUTPUT_DIR / 'model.pkl'
METRICS_OUTPUT = OUTPUT_DIR / 'metrics.json'
MODEL_PIPELINE_OUTPUT = OUTPUT_DIR / 'final_model_pipeline.joblib'

print('DATA_DIR:', DATA_DIR)
print('SNAPSHOT_FILE exists:', SNAPSHOT_FILE.exists())
print('LABEL_FILE exists:', LABEL_FILE.exists())

DATA_DIR: E:\d2c-churn-modeling\data
SNAPSHOT_FILE exists: True
LABEL_FILE exists: True


In [51]:
snapshot_df = pd.read_csv(SNAPSHOT_FILE)
labels_df = pd.read_csv(LABEL_FILE)
print('Snapshot shape:', snapshot_df.shape)
print('Labels shape:', labels_df.shape)
print('Snapshot columns:', snapshot_df.columns.tolist())
print('Labels columns:', labels_df.columns.tolist())
snapshot_df.head()

Snapshot shape: (2400, 29)
Labels shape: (2400, 4)
Snapshot columns: ['customer_id', 'snapshot_date', 'city_tier', 'age_group', 'acquisition_channel', 'loyalty_tier', 'preferred_category', 'marketing_consent', 'recency_days', 'frequency_180d', 'monetary_180d', 'return_rate_180d', 'avg_discount_pct_180d', 'avg_rating_180d', 'category_diversity_180d', 'ticket_count_90d', 'negative_ticket_rate_90d', 'avg_resolution_hours_90d', 'days_since_signup', 'sessions_30d', 'product_views_30d', 'cart_adds_30d', 'wishlist_adds_30d', 'abandoned_carts_30d', 'email_opens_30d', 'campaign_clicks_30d', 'last_visit_days_ago', 'churn_next_60d', 'split']
Labels columns: ['customer_id', 'snapshot_date', 'churn_next_60d', 'split']


,customer_id,snapshot_date,city_tier,age_group,acquisition_channel,loyalty_tier,preferred_category,marketing_consent,recency_days,frequency_180d,monetary_180d,return_rate_180d,avg_discount_pct_180d,avg_rating_180d,category_diversity_180d,ticket_count_90d,negative_ticket_rate_90d,avg_resolution_hours_90d,days_since_signup,sessions_30d,product_views_30d,cart_adds_30d,wishlist_adds_30d,abandoned_carts_30d,email_opens_30d,campaign_clicks_30d,last_visit_days_ago,churn_next_60d,split
0,CUST00001,2025-09-30,Tier 1,18-24,Instagram,Silver,Makeup,Yes,107,1,362.73,0.0,0.23,3.0,1,0,0.0,0.0,524,1,4,0,0,0,2,0,20,1,train
1,CUST00002,2025-09-30,Tier 2,25-34,Marketplace,Silver,Hair Care,Yes,40,1,581.00,0.0,0.23,4.0,1,1,0.0,1.0,121,8,31,4,2,3,0,0,0,0,train
2,CUST00003,2025-09-30,Tier 1,25-34,Influencer,NaN,Skin Care,Yes,171,1,649.98,0.0,0.47,2.0,1,0,0.0,0.0,206,1,3,0,0,0,0,0,26,1,train
3,CUST00004,2025-09-30,Tier 3,25-34,Google Search,NaN,Fragrance,No,131,1,1604.04,0.0,0.16,2.0,1,0,0.0,0.0,168,1,6,0,0,0,0,0,14,1,train
4,CUST00005,2025-09-30,Tier 3,35-44,Organic,Gold,Hair Care,Yes,38,3,1781.90,0.0,0.48,1.0,2,0,0.0,0.0,405,18,95,4,1,1,3,1,9,0,train


## Leakage checks

We use the pre-built snapshot file `rfm_modeling_snapshot.csv`, which contains only information available on or before the snapshot date. The target labels are loaded separately from `churn_labels.csv`, which provides the churn outcome and the train/validation/test split. This ensures no post-snapshot future information is used as model input.

In [52]:
# Robust merge and leakage detection
# Drop any pre-existing label/split columns from the snapshot to avoid accidental leakage
for _col in ['churn_next_60d', 'split']:
    if _col in snapshot_df.columns:
        print(f"Warning: dropping pre-existing column '{_col}' from snapshot to avoid merge conflicts/leakage")

snapshot_clean = snapshot_df.drop(columns=[c for c in ['churn_next_60d', 'split'] if c in snapshot_df.columns]).copy()

# Prepare labels and check duplicates
labels_subset = labels_df[['customer_id', 'churn_next_60d', 'split']].drop_duplicates('customer_id')
dup_mask = labels_subset['customer_id'].duplicated()
if dup_mask.any():
    raise ValueError(f"Duplicate customer_id entries found in labels for {dup_mask.sum()} rows; fix labels file before proceeding.")

# Merge with one-to-one validation
merged_df = snapshot_clean.merge(
    labels_subset,
    on='customer_id',
    how='left',
    validate='one_to_one',
)

missing_labels = merged_df['churn_next_60d'].isna().sum()
if missing_labels > 0:
    example_ids = merged_df.loc[merged_df['churn_next_60d'].isna(), 'customer_id'].tolist()[:10]
    raise ValueError(f"Missing churn labels for {missing_labels} customers (examples: {example_ids}). Aborting until labels fixed.")

print('Merged shape:', merged_df.shape)
print('Train/validation/test distribution:')
print(merged_df['split'].value_counts(dropna=False))
print('Churn rate overall:', merged_df['churn_next_60d'].mean())

excluded_columns = {'customer_id', 'snapshot_date', 'churn_next_60d', 'split'}
leakage_columns = [col for col in merged_df.columns
                   if col not in excluded_columns and any(k in col.lower() for k in ['future', 'post', 'next_', 'after_snapshot'])]
print('Potential leakage columns:', leakage_columns)


Merged shape: (2400, 29)
Train/validation/test distribution:
split
train         1728
validation     336
test           336
Name: count, dtype: int64
Churn rate overall: 0.46958333333333335
Potential leakage columns: []


In [53]:
# Extended leakage checks: verify snapshot dates, source files and post-snapshot rows
import warnings

# determine snapshot date
try:
    snap_dates = pd.to_datetime(snapshot_df['snapshot_date'].unique())
    if len(snap_dates) != 1:
        warnings.warn(f"Multiple snapshot_date values found: {snap_dates}")
    snapshot_date = snap_dates[0]
    print('Snapshot reference date:', snapshot_date.date())
except Exception:
    snapshot_date = None
    warnings.warn('Could not infer a single snapshot_date from snapshot_df')

# check labels snapshot_date
try:
    label_snap_dates = pd.to_datetime(labels_df['snapshot_date'].unique())
    if len(label_snap_dates) != 1:
        warnings.warn(f"Multiple label snapshot_date values found: {label_snap_dates}")
    else:
        if snapshot_date is not None and label_snap_dates[0] != snapshot_date:
            raise ValueError(f"Snapshot date mismatch: snapshot has {snapshot_date.date()} but labels have {label_snap_dates[0].date()}")
    print('Label snapshot date OK')
except Exception as e:
    warnings.warn(f'Label snapshot date check failed: {e}')

# check orders for post-snapshot rows (must not be used as features)
orders_path = DATA_DIR / 'orders.csv'
if orders_path.exists():
    orders = pd.read_csv(orders_path, parse_dates=['order_date'])
    if snapshot_date is not None:
        post_snapshot = orders[orders['order_date'] > snapshot_date]
        print('Orders rows after snapshot:', len(post_snapshot))
        if len(post_snapshot) > 0:
            warnings.warn('orders.csv contains post-snapshot rows. Do NOT use post-snapshot orders as features.')
else:
    print('orders.csv not found; skipping orders post-snapshot check')

# check support_tickets dates
support_path = DATA_DIR / 'support_tickets.csv'
if support_path.exists():
    tickets = pd.read_csv(support_path, parse_dates=['ticket_date'])
    if snapshot_date is not None:
        bad = tickets[tickets['ticket_date'] > snapshot_date]
        print('Support tickets after snapshot:', len(bad))
        if len(bad) > 0:
            warnings.warn('support_tickets.csv contains rows after the snapshot date')
else:
    print('support_tickets.csv not found; skipping')

# check web events snapshot
web_path = DATA_DIR / 'web_events_snapshot.csv'
if web_path.exists():
    web = pd.read_csv(web_path, parse_dates=['snapshot_date'])
    unique_web_dates = web['snapshot_date'].unique()
    print('web_events_snapshot snapshot_date unique values:', unique_web_dates)
    if snapshot_date is not None and any(pd.to_datetime(unique_web_dates) != snapshot_date):
        warnings.warn('web_events_snapshot contains snapshot_date values different from the main snapshot')
else:
    print('web_events_snapshot.csv not found; skipping')

# check for suspicious column names that might indicate leakage
suspicious = [c for c in snapshot_df.columns if any(k in c.lower() for k in ['future', 'post', 'next_', 'after_snapshot'])]
print('Suspicious column names indicating potential leakage:', suspicious)

print('\nExtended leakage checks complete.')


Snapshot reference date: 2025-09-30
Label snapshot date OK
Orders rows after snapshot: 1872
Support tickets after snapshot: 0
web_events_snapshot snapshot_date unique values: <DatetimeArray>
['2025-09-30 00:00:00']
Length: 1, dtype: datetime64[ns]
Suspicious column names indicating potential leakage: ['churn_next_60d']

Extended leakage checks complete.


C:\Users\Admin\AppData\Local\Temp\ipykernel_12228\1143767173.py:35: UserWarning: orders.csv contains post-snapshot rows. Do NOT use post-snapshot orders as features.
  warnings.warn('orders.csv contains post-snapshot rows. Do NOT use post-snapshot orders as features.')


In [54]:
feature_cols = [
    col for col in merged_df.columns
    if col not in ['customer_id', 'snapshot_date', 'churn_next_60d', 'split']
]
numeric_features = [col for col in feature_cols if pd.api.types.is_numeric_dtype(merged_df[col])]
categorical_features = [col for col in feature_cols if pd.api.types.is_object_dtype(merged_df[col])]
print('Numeric features:', numeric_features)
print('Categorical features:', categorical_features)
print('Total features:', len(feature_cols))

Numeric features: ['recency_days', 'frequency_180d', 'monetary_180d', 'return_rate_180d', 'avg_discount_pct_180d', 'avg_rating_180d', 'category_diversity_180d', 'ticket_count_90d', 'negative_ticket_rate_90d', 'avg_resolution_hours_90d', 'days_since_signup', 'sessions_30d', 'product_views_30d', 'cart_adds_30d', 'wishlist_adds_30d', 'abandoned_carts_30d', 'email_opens_30d', 'campaign_clicks_30d', 'last_visit_days_ago']
Categorical features: ['city_tier', 'age_group', 'acquisition_channel', 'loyalty_tier', 'preferred_category', 'marketing_consent']
Total features: 25


In [55]:
train_df = merged_df[merged_df['split'] == 'train'].reset_index(drop=True)
valid_df = merged_df[merged_df['split'] == 'validation'].reset_index(drop=True)
test_df = merged_df[merged_df['split'] == 'test'].reset_index(drop=True)
print('Train shape:', train_df.shape)
print('Validation shape:', valid_df.shape)
print('Test shape:', test_df.shape)

Train shape: (1728, 29)
Validation shape: (336, 29)
Test shape: (336, 29)


## Preprocessing pipelines

Numeric features are median-imputed and scaled. Categorical features are imputed with a placeholder and one-hot encoded.

In [56]:
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features),
])

In [57]:
X_train = train_df[feature_cols]
y_train = train_df['churn_next_60d'].astype(int)
X_valid = valid_df[feature_cols]
y_valid = valid_df['churn_next_60d'].astype(int)
X_test = test_df[feature_cols]
y_test = test_df['churn_next_60d'].astype(int)

## Baseline model: Logistic Regression

In [58]:
X_train = train_df[feature_cols]
y_train = train_df['churn_next_60d'].astype(int)
X_valid = valid_df[feature_cols]
y_valid = valid_df['churn_next_60d'].astype(int)
X_test = test_df[feature_cols]
y_test = test_df['churn_next_60d'].astype(int)

baseline = Pipeline([
    ('preprocessor', preprocessor),
    ('clf', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)),
])
baseline.fit(X_train, y_train)
valid_probs_baseline = baseline.predict_proba(X_valid)[:, 1]
valid_preds_baseline = (valid_probs_baseline >= 0.5).astype(int)
baseline_metrics = {
    'accuracy': accuracy_score(y_valid, valid_preds_baseline),
    'precision': precision_score(y_valid, valid_preds_baseline, zero_division=0),
    'recall': recall_score(y_valid, valid_preds_baseline, zero_division=0),
    'f1_score': f1_score(y_valid, valid_preds_baseline, zero_division=0),
    'roc_auc': roc_auc_score(y_valid, valid_probs_baseline),
}
baseline_metrics

{'accuracy': 0.8154761904761905,
 'precision': 0.7972027972027972,
 'recall': 0.7755102040816326,
 'f1_score': 0.7862068965517242,
 'roc_auc': np.float64(0.8825900730662636)}

## Stronger model: Random Forest

In [59]:
final_model = Pipeline([
    ('preprocessor', preprocessor),
    ('clf', RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced'))
])
final_model.fit(X_train, y_train)
valid_probs_final = final_model.predict_proba(X_valid)[:, 1]
valid_preds_final = (valid_probs_final >= 0.5).astype(int)
final_metrics = {
    'accuracy': accuracy_score(y_valid, valid_preds_final),
    'precision': precision_score(y_valid, valid_preds_final, zero_division=0),
    'recall': recall_score(y_valid, valid_preds_final, zero_division=0),
    'f1_score': f1_score(y_valid, valid_preds_final, zero_division=0),
    'roc_auc': roc_auc_score(y_valid, valid_probs_final),
}
final_metrics

{'accuracy': 0.7916666666666666,
 'precision': 0.7851851851851852,
 'recall': 0.7210884353741497,
 'f1_score': 0.75177304964539,
 'roc_auc': np.float64(0.8672389590756938)}

## Threshold selection

In [60]:
precision, recall, thresholds = precision_recall_curve(y_valid, valid_probs_final)
threshold_df = pd.DataFrame({
    'threshold': thresholds,
    'precision': precision[:-1],
    'recall': recall[:-1],
})
threshold_df.head()

,threshold,precision,recall
0,0.015,0.437500,1.0
1,0.020,0.438806,1.0
2,0.025,0.440120,1.0
3,0.030,0.442771,1.0
4,0.035,0.445455,1.0


In [61]:
preferred_threshold = threshold_df.loc[threshold_df['precision'] >= 0.3, 'threshold']
if len(preferred_threshold) > 0:
    selected_threshold = float(preferred_threshold.iloc[threshold_df.loc[threshold_df['precision'] >= 0.3, 'recall'].idxmax()])
else:
    selected_threshold = float(threshold_df.loc[threshold_df['recall'].idxmax(), 'threshold'])
selected_threshold

0.015

In [62]:
test_probs_final = final_model.predict_proba(X_test)[:, 1]
test_preds_final = (test_probs_final >= selected_threshold).astype(int)
test_metrics = {
    'accuracy': accuracy_score(y_test, test_preds_final),
    'precision': precision_score(y_test, test_preds_final, zero_division=0),
    'recall': recall_score(y_test, test_preds_final, zero_division=0),
    'f1_score': f1_score(y_test, test_preds_final, zero_division=0),
    'roc_auc': roc_auc_score(y_test, test_probs_final),
    'confusion_matrix': confusion_matrix(y_test, test_preds_final).tolist(),
    'threshold': selected_threshold,
}
test_metrics

{'accuracy': 0.5,
 'precision': 0.5,
 'recall': 1.0,
 'f1_score': 0.6666666666666666,
 'roc_auc': np.float64(0.8809346655328798),
 'confusion_matrix': [[0, 168], [0, 168]],
 'threshold': 0.015}

## Feature importance and interpretability

In [63]:
final_clf = final_model.named_steps['clf']
feature_names = numeric_features
onehot = final_model.named_steps['preprocessor'].named_transformers_['cat'].named_steps['onehot']
feature_names += onehot.get_feature_names_out(categorical_features).tolist()
importances = final_clf.feature_importances_
importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importances,
}).sort_values('importance', ascending=False).reset_index(drop=True)
importance_df.head(15)

,feature,importance
0,recency_days,0.188668
1,last_visit_days_ago,0.147037
2,monetary_180d,0.081752
3,days_since_signup,0.054428
4,product_views_30d,0.053875
5,avg_discount_pct_180d,0.047289
6,frequency_180d,0.039528
7,sessions_30d,0.037125
8,category_diversity_180d,0.028575
9,avg_rating_180d,0.026886


In [64]:
# combine train + validation using the original dataframes to ensure expected columns exist
X_combined = pd.concat([train_df[feature_cols], valid_df[feature_cols]], axis=0).reset_index(drop=True)
y_combined = pd.concat([train_df['churn_next_60d'].astype(int), valid_df['churn_next_60d'].astype(int)], axis=0).reset_index(drop=True)

# make preprocessor robust to any missing columns by using the intersection with X_combined
present_feature_cols = [c for c in feature_cols if c in X_combined.columns]
numeric_features_combined = [c for c in present_feature_cols if pd.api.types.is_numeric_dtype(X_combined[c])]
categorical_features_combined = [c for c in present_feature_cols if pd.api.types.is_object_dtype(X_combined[c])]

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features_combined),
    ('cat', categorical_transformer, categorical_features_combined),
])

final_model = Pipeline([
    ('preprocessor', preprocessor),
    ('clf', RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced'))
])

final_model.fit(X_combined[present_feature_cols], y_combined)

joblib.dump(final_model, MODEL_OUTPUT)
joblib.dump(final_model, MODEL_PIPELINE_OUTPUT)

metrics = {
    'baseline_validation': baseline_metrics,
    'final_validation': final_metrics,
    'final_test': test_metrics,
    'selected_threshold': float(selected_threshold),
}

# ensure JSON-serializable (convert numpy scalars)
metrics_json = json.dumps(metrics, default=lambda o: o.item() if hasattr(o, 'item') else o, indent=2)
with open(METRICS_OUTPUT, 'w', encoding='utf-8') as f:
    f.write(metrics_json)

print('Model saved to', MODEL_OUTPUT)
print('Pipeline saved to', MODEL_PIPELINE_OUTPUT)
print('Metrics saved to', METRICS_OUTPUT)

Model saved to E:\d2c-churn-modeling\model.pkl
Pipeline saved to E:\d2c-churn-modeling\final_model_pipeline.joblib
Metrics saved to E:\d2c-churn-modeling\metrics.json
